# Pipeline combinado (provisional): U-Net + limpieza morfológica + HSV/Lab

Notebook de trabajo para ir armando, bloque por bloque, un pipeline que combine:
1. Predicción de agua con la U-Net entrenada (con su IoU)
2. Limpieza morfológica de la máscara (cerrar huecos, suavizar bordes)
3. Refinamiento con el segmentador clásico HSV/Lab de `water_masc1`

**Por ahora** las imágenes se buscan por nombre dentro de `river_water_index.csv` (así podemos calcular IoU contra su máscara real). Más adelante se adapta para cargar cualquier imagen local, tenga o no máscara de referencia.

## Configuración

In [ ]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

BASE_PATH = Path(r"D:\proyecto_eutrofizacion")
sys.path.insert(0, str(BASE_PATH / "src"))

from unet.config import UNetConfig
from unet.preprocessing import load_image_rgb, load_mask_binary, downscale_image, downscale_mask
from unet.patches import get_patch_positions
from unet.reconstruction import reconstruct_full_prediction
from unet.augmentation import get_val_transforms
from unet.metrics import compute_metrics
from unet.inference import load_model

from water_masc1 import segmentar_agua
from water_masc2 import conectar_graffiti_y_cerrar, suavizar_bordes_contorno

# NOTA: no importamos unet.visualization aquí a propósito -- ese módulo fuerza
# el backend "Agg" de matplotlib (pensado para guardar figuras sin pantalla
# durante el entrenamiento), lo que rompería la visualización inline en este
# notebook. Usamos nuestro propio helper de plotting más abajo.

config = UNetConfig(base_path=BASE_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df_river = pd.read_csv(config.get_path(config.csv_path))
print(f"Índice cargado: {len(df_river)} imágenes con máscara disponible")


In [ ]:
def buscar_imagen_por_nombre(nombre, df=df_river):
    """Busca una imagen por nombre de archivo en el índice del dataset.
    Devuelve la fila del CSV (con filepath y segmentation_mask_path)."""
    filas = df[df["filepath"].str.contains(nombre, regex=False)]
    if filas.empty:
        raise ValueError(f"No se encontró la imagen '{nombre}' en el índice.")
    return filas.iloc[0]


def mostrar_paneles(image_ds, paneles, suptitle=None):
    """Muestra la imagen original + N máscaras/paneles lado a lado.
    paneles: lista de tuplas (titulo, array, cmap).
    """
    n = len(paneles) + 1
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    axes[0].imshow(image_ds)
    axes[0].set_title("Imagen")
    axes[0].axis("off")

    for ax, (titulo, arr, cmap) in zip(axes[1:], paneles):
        ax.imshow(arr, cmap=cmap)
        ax.set_title(titulo)
        ax.axis("off")

    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()


# Diccionario donde se van acumulando los resultados de cada imagen a medida
# que avanzamos por los bloques (cada bloque le añade sus propias claves).
resultados = {}

print("Listo: buscar_imagen_por_nombre(), mostrar_paneles() y 'resultados' definidos.")


## Predicción con U-Net (IoU)

Recibe una lista de nombres de imagen (por ahora, deben existir en `river_water_index.csv` para poder comparar contra su máscara real). Predice la máscara de agua con la U-Net y calcula el IoU de cada una.

In [ ]:
# === IMÁGENES A PROCESAR ===
# Cambia estos nombres por los que quieras probar (deben existir en el índice
# del dataset). Cuando adaptemos el pipeline para cargar imágenes sueltas del
# local, esta lista dejará de depender de river_water_index.csv.
IMAGENES = [
    "DJI_10226.JPG",
    "DJI_11677.JPG",
    "DJI_2305.JPG",
    "DJI_10674.JPG",
    "DJI_9805.JPG",
]

# --- Modelo ---
checkpoint_path = config.get_path(config.checkpoints_dir) / "best_model.pth"
transform = get_val_transforms(config.imagenet_mean, config.imagenet_std)

model = load_model(
    checkpoint_path,
    encoder_name=config.encoder_name,
    encoder_weights=None,
    device=device,
)

# --- Predicción por imagen ---
for nombre in IMAGENES:
    fila = buscar_imagen_por_nombre(nombre)
    img_path = BASE_PATH / fila["filepath"]
    mask_path = BASE_PATH / fila["segmentation_mask_path"]

    image = load_image_rgb(img_path)
    h, w = image.shape[:2]
    mask_gt = load_mask_binary(mask_path, h, w, BASE_PATH)

    image_ds = downscale_image(image, config.downscale_factor)
    mask_gt_ds = downscale_mask(mask_gt, config.downscale_factor)
    h_ds, w_ds = image_ds.shape[:2]

    positions = get_patch_positions(h_ds, w_ds, config.patch_size, config.stride)
    prob_avg, mask_unet = reconstruct_full_prediction(
        model=model,
        image_ds=image_ds,
        positions=positions,
        transform=transform,
        patch_size=config.patch_size,
        threshold=config.threshold,
        batch_size=config.batch_size * 2,
        device=device,
    )

    gt_float = (mask_gt_ds > 127).astype(np.float32)
    m = compute_metrics(prob_avg, gt_float, threshold=config.threshold)

    resultados[nombre] = {
        "image_ds": image_ds,
        "mask_gt_ds": mask_gt_ds,
        "mask_unet": mask_unet,
        "iou_unet": m.iou,
        "dice_unet": m.dice,
    }

    print(f"{nombre}: {m}")

# --- Visualización ---
for nombre, r in resultados.items():
    mostrar_paneles(
        r["image_ds"],
        [
            ("Ground truth", r["mask_gt_ds"], "gray"),
            ("Predicción U-Net", r["mask_unet"], "gray"),
        ],
        suptitle=f"{nombre}  —  IoU={r['iou_unet']:.4f}  |  Dice={r['dice_unet']:.4f}",
    )


## Cerrar máscaras

Aplica `conectar_graffiti_y_cerrar` (cierra huecos/gaps pequeños) y `suavizar_bordes_contorno` (suaviza el contorno) sobre la máscara que predijo la U-Net.

**Ojo**: `suavizar_bordes_contorno` se queda solo con el contorno EXTERNO más grande — si la máscara tiene varias zonas de agua desconectadas entre sí, las más pequeñas se pierden. Lo dejamos así por ahora porque es la función que ya tenías; lo ajustamos si al ver los resultados no conviene.

In [ ]:
for nombre, r in resultados.items():
    mask_cerrada = conectar_graffiti_y_cerrar(
        r["mask_unet"], skeleton_kernel=3, dilation_kernel=30
    )
    mask_cerrada = suavizar_bordes_contorno(mask_cerrada, contour_approx=7)

    gt_float = (r["mask_gt_ds"] > 127).astype(np.float32)
    m = compute_metrics(mask_cerrada, gt_float, threshold=0.5)

    r["mask_cerrada"] = mask_cerrada
    r["iou_cerrada"] = m.iou
    r["dice_cerrada"] = m.dice

    print(f"{nombre}: IoU antes={r['iou_unet']:.4f}  ->  después de cerrar={m.iou:.4f}")

# --- Visualización ---
for nombre, r in resultados.items():
    mostrar_paneles(
        r["image_ds"],
        [
            ("U-Net (antes)", r["mask_unet"], "gray"),
            ("Cerrada (después)", r["mask_cerrada"], "gray"),
        ],
        suptitle=(
            f"{nombre}  —  IoU antes={r['iou_unet']:.4f}  "
            f"->  después={r['iou_cerrada']:.4f}"
        ),
    )


## Parámetros HSV y demás

Genera una segunda máscara usando **los canales a\* (Lab, rojo-verde) y L (Lab, iluminancia)** —no el `segmentar_agua` completo con b*/H/S— combinados entre sí con AND, y el resultado se combina con **intersección (AND)** con la máscara del bloque anterior (`mask_cerrada`). AND es más conservador que el OR que usábamos antes: un píxel solo queda como agua si los métodos coinciden en que lo es.

Si en realidad querías combinar con la máscara cruda de la U-Net (antes de "Cerrar máscaras"), cambia `r["mask_cerrada"]` por `r["mask_unet"]` en la celda de abajo.

In [ ]:
def segmentar_agua_canales_a_l(imagen_bgr, a_min, a_max, l_min, l_max, downscale=4):
    """
    Máscaras de agua usando los canales a* (Lab, eje rojo-verde) y L (Lab,
    iluminancia), por separado y combinados con AND. Es el mismo paso que
    hace water_masc1.segmentar_agua internamente para a* y L, pero sin
    combinarlo con b*, H ni S.

    Devuelve (mask_a, mask_l, mask_a_l) en la resolución ORIGINAL de la
    imagen (0/255, uint8), igual que segmentar_agua.
    """
    altura_orig, ancho_orig = imagen_bgr.shape[:2]

    if downscale > 1:
        imagen_small = cv2.resize(
            imagen_bgr,
            (ancho_orig // downscale, altura_orig // downscale),
            interpolation=cv2.INTER_AREA,
        )
    else:
        imagen_small = imagen_bgr

    lab = cv2.cvtColor(imagen_small, cv2.COLOR_BGR2Lab)
    l_channel = lab[:, :, 0].astype(np.float32)         # ya en escala 0-255
    a_channel = lab[:, :, 1].astype(np.float32) - 128    # centrado en 0

    mask_a_bool = (a_channel >= a_min) & (a_channel <= a_max)
    mask_l_bool = (l_channel >= l_min) & (l_channel <= l_max)

    mask_a = mask_a_bool.astype(np.uint8) * 255
    mask_l = mask_l_bool.astype(np.uint8) * 255
    mask_a_l = (mask_a_bool & mask_l_bool).astype(np.uint8) * 255

    if downscale > 1:
        size = (ancho_orig, altura_orig)
        mask_a = cv2.resize(mask_a, size, interpolation=cv2.INTER_NEAREST)
        mask_l = cv2.resize(mask_l, size, interpolation=cv2.INTER_NEAREST)
        mask_a_l = cv2.resize(mask_a_l, size, interpolation=cv2.INTER_NEAREST)

    return mask_a, mask_l, mask_a_l


# Mismos límites de a* y L que en PARAMETROS de 01_water_segmentation.ipynb
A_MIN, A_MAX = -120, 1
L_MIN, L_MAX = 50, 255

for nombre, r in resultados.items():
    fila = buscar_imagen_por_nombre(nombre)
    ruta_imagen = BASE_PATH / fila["filepath"]

    # cv2.imread carga BGR de forma nativa -- igual que espera la función
    imagen_bgr = cv2.imread(str(ruta_imagen))

    mask_a_orig, mask_l_orig, mask_al_orig = segmentar_agua_canales_a_l(
        imagen_bgr, a_min=A_MIN, a_max=A_MAX, l_min=L_MIN, l_max=L_MAX, downscale=4
    )

    # Las bajamos al mismo tamaño (x2) que usa la U-Net para poder combinarlas
    h_ds, w_ds = r["mask_cerrada"].shape
    mask_a = cv2.resize(mask_a_orig, (w_ds, h_ds), interpolation=cv2.INTER_NEAREST)
    mask_l = cv2.resize(mask_l_orig, (w_ds, h_ds), interpolation=cv2.INTER_NEAREST)
    mask_a_l = cv2.resize(mask_al_orig, (w_ds, h_ds), interpolation=cv2.INTER_NEAREST)

    mask_combinada = cv2.bitwise_and(r["mask_cerrada"], mask_a_l)

    gt_float = (r["mask_gt_ds"] > 127).astype(np.float32)
    m_a = compute_metrics(mask_a, gt_float, threshold=0.5)
    m_l = compute_metrics(mask_l, gt_float, threshold=0.5)
    m_a_l = compute_metrics(mask_a_l, gt_float, threshold=0.5)
    m_comb = compute_metrics(mask_combinada, gt_float, threshold=0.5)

    r["mask_a"] = mask_a
    r["mask_l"] = mask_l
    r["mask_a_l"] = mask_a_l
    r["mask_combinada"] = mask_combinada
    r["iou_a"] = m_a.iou
    r["iou_l"] = m_l.iou
    r["iou_a_l"] = m_a_l.iou
    r["iou_combinada"] = m_comb.iou

    print(
        f"{nombre}: cerrada={r['iou_cerrada']:.4f}  |  a*={m_a.iou:.4f}  |  "
        f"L={m_l.iou:.4f}  |  a*+L={m_a_l.iou:.4f}  |  combinada (AND)={m_comb.iou:.4f}"
    )

# --- Visualización ---
for nombre, r in resultados.items():
    mostrar_paneles(
        r["image_ds"],
        [
            ("Cerrada (U-Net)", r["mask_cerrada"], "gray"),
            ("Canal a*", r["mask_a"], "gray"),
            ("Canal L", r["mask_l"], "gray"),
            ("a* AND L", r["mask_a_l"], "gray"),
            ("Combinada final", r["mask_combinada"], "gray"),
        ],
        suptitle=(
            f"{nombre}  —  cerrada={r['iou_cerrada']:.4f}  |  a*={r['iou_a']:.4f}  |  "
            f"L={r['iou_l']:.4f}  |  a*+L={r['iou_a_l']:.4f}  |  final={r['iou_combinada']:.4f}"
        ),
    )
